# ガイドカメラ(Atik314L+)に天体導入後、赤道儀による追尾を補助するプログラムを試作

In [19]:
import os
import sys
from pathlib import Path
import win32com.client
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import plotly.express as px
import astropy


# 1. initial-parameter setting

In [15]:
# time information
observation_date = '20250301' # 観測日

# path information
scat_auto_obserbation_system_path = Path().absolute().parent
temporal_images_path = scat_auto_obserbation_system_path / '99_temporal_images'
slit_guide_camera_path = temporal_images_path / '02_slit_guide_camera'
slit_guide_camera_obs_date_path = slit_guide_camera_path / observation_date
src_path = temporal_images_path / 'src'

# camera information
slit_guide_camera_name = 'Atik314L+' # ガイドカメラ名
slit_guide_camera_exposure_time = 0.05 # ガイドカメラ露光時間(秒)
slit_guide_camera_cooler_templatur = -5 # ガイドカメラ冷却温度
slit_center_position_pixcel = [1024, 1024] # ガイドカメラ中心位置(ピクセル) # test value


In [16]:
if not slit_guide_camera_obs_date_path.exists():
    print(f'{observation_date} directory does not exist. Creating...')
    slit_guide_camera_obs_date_path.mkdir(parents=True, exist_ok=True)
    print(f'{observation_date} directory created')
else:
    print(f'{observation_date} directory already exists')

20250301 directory does not exist. Creating...
20250301 directory created


# 2. capturing image of slit-guide camera

In [20]:
camera = win32com.client.Dispatch("AtikCameras.AtikCamera") # Atik314L+カメラをマウント


com_error: (-2147221005, 'クラス文字列が無効です', None, None)

In [27]:
import win32com.client
import pythoncom
import time

def mount_camera_by_clsid(clsid):
    """
    指定されたCLSIDを使用してカメラをマウントする
    Args:
        clsid (str): カメラのCLSID
    """
    print(f"Attempting to mount camera with CLSID: {clsid}")
    
    try:
        # CLSIDを使用してCOMオブジェクトを作成
        camera = win32com.client.Dispatch(clsid)
        print("Successfully created camera object")
        
        # 利用可能なメソッドとプロパティを表示
        methods = [item for item in dir(camera) if not item.startswith('_')]
        print("\nAvailable methods and properties:")
        for method in methods:
            print(f"- {method}")
        
        # カメラの接続状態を確認（可能な場合）
        try:
            if hasattr(camera, 'Connected'):
                print(f"\nCamera connected: {camera.Connected}")
            if hasattr(camera, 'Name'):
                print(f"Camera name: {camera.Name}")
            if hasattr(camera, 'Description'):
                print(f"Camera description: {camera.Description}")
        except Exception as e:
            print(f"Error checking camera status: {e}")
        
        return camera
        
    except Exception as e:
        print(f"Error mounting camera: {e}")
        return None

def main():
    # Atikカメラの既知のCLSID
    camera_clsids = [
        "{36fc9e60-c465-11cf-8056-444553540000}",  # 元のCLSID
        # "{DE9CEEA7-F71C-43A4-A862-FC5EE185C25E}",  # 別のよく使われるCLSID
    ]
    
    camera = None
    for clsid in camera_clsids:
        print(f"\nTrying CLSID: {clsid}")
        camera = mount_camera_by_clsid(clsid)
        if camera:
            print("Camera mounted successfully!")
            break
        else:
            print("Failed to mount with this CLSID, trying next...")
    
    if camera:
        try:
            # カメラとの接続テスト（実際のメソッドはカメラのAPIに依存）
            print("\nTesting camera connection...")
            # 例: camera.Connect() や camera.StartExposure() など
            # 実際のメソッドはカメラのドキュメントを参照してください
            
            time.sleep(2)  # カメラの初期化を待つ
            
        except Exception as e:
            print(f"Error testing camera: {e}")
        finally:
            try:
                # 必要に応じてクリーンアップ
                # 例: camera.Disconnect()
                pass
            except:
                pass
    else:
        print("\nFailed to mount camera with any known CLSID")
        print("Please check:")
        print("1. Camera is physically connected")
        print("2. Camera drivers are installed")
        print("3. Running script with administrator privileges")
        print("4. Camera software is properly installed")

if __name__ == "__main__":
    main()


Trying CLSID: {36fc9e60-c465-11cf-8056-444553540000}
Attempting to mount camera with CLSID: {36fc9e60-c465-11cf-8056-444553540000}
Error mounting camera: (-2147221164, 'クラスが登録されていません', None, None)
Failed to mount with this CLSID, trying next...

Failed to mount camera with any known CLSID
Please check:
1. Camera is physically connected
2. Camera drivers are installed
3. Running script with administrator privileges
4. Camera software is properly installed


In [21]:
import pythoncom

# 登録されているCOMオブジェクトを表示
def list_com_objects():
    try:
        from win32com.client import gencache
        objects = gencache.GetGeneratedInfos()
        for obj in objects:
            print(f"Registered COM object: {obj}")
    except Exception as e:
        print(f"Error listing COM objects: {e}")

list_com_objects()

In [23]:
from win32com.client import gencache
gencache.GetGeneratedInfos()

[]

In [24]:
import winreg

def list_com_objects_from_registry():
    # CLSID キーを開く
    with winreg.OpenKey(winreg.HKEY_CLASSES_ROOT, 'CLSID') as key:
        i = 0
        while True:
            try:
                # CLSIDの列挙
                clsid = winreg.EnumKey(key, i)
                # 各CLSIDの詳細情報を取得
                with winreg.OpenKey(key, clsid) as clsid_key:
                    try:
                        # デフォルト値（説明）を取得
                        value, _ = winreg.QueryValueEx(clsid_key, "")
                        print(f"CLSID: {clsid}")
                        print(f"Description: {value}")
                        print("-" * 50)
                    except WindowsError:
                        pass
                i += 1
            except WindowsError:
                break

if __name__ == "__main__":
    list_com_objects_from_registry()

CLSID: CLSID
Description: {0000031A-0000-0000-C000-000000000046}
--------------------------------------------------
CLSID: {0000002F-0000-0000-C000-000000000046}
Description: CLSID_RecordInfo
--------------------------------------------------
CLSID: {00000300-0000-0000-C000-000000000046}
Description: StdOleLink
--------------------------------------------------
CLSID: {00000301-A8F2-4877-BA0A-FD2B6645FB94}
Description: PSFactoryBuffer
--------------------------------------------------
CLSID: {00000303-0000-0000-C000-000000000046}
Description: FileMoniker
--------------------------------------------------
CLSID: {00000304-0000-0000-C000-000000000046}
Description: ItemMoniker
--------------------------------------------------
CLSID: {00000305-0000-0000-C000-000000000046}
Description: AntiMoniker
--------------------------------------------------
CLSID: {00000306-0000-0000-C000-000000000046}
Description: PointerMoniker
--------------------------------------------------
CLSID: {00000308-00

In [ ]:
camera.FindDevices() # 利用可能なカメラを検索

In [ ]:
import numpy as np
import polars as pl
import os
from pathlib import Path
from datetime import datetime
import win32com.client
import time

# 観測日のディレクトリパスを設定
current_path = Path().absolute()
today = datetime.now().strftime("%Y%m%d")
obs_path = current_path / "data" / today

# ディレクトリが存在しない場合は作成
if not obs_path.exists():
    obs_path.mkdir(parents=True, exist_ok=True)

try:
    # Atik314L+カメラをマウント
    camera = win32com.client.Dispatch("AtikCameras.AtikCamera")
    
    # 利用可能なカメラを検索
    camera.FindDevices()
    
    # 最初に見つかったカメラに接続
    camera.Connect()
    print("Camera connected successfully")
    
    # 露出設定
    exposure_time = 1.0  # 露出時間（秒）
    binning = 1         # ビニング設定（1x1）
    
    # 撮影時刻を含むファイル名を生成
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    image_filename = f"image_{timestamp}.fits"
    image_path = obs_path / image_filename
    
    # 撮影開始
    print(f"Starting exposure: {exposure_time} seconds")
    camera.StartExposure(exposure_time, binning)
    
    # 露出完了まで待機
    while not camera.ImageReady:
        time.sleep(0.1)
    
    # 画像を取得して保存
    camera.SaveImage(str(image_path))
    print(f"Image saved to: {image_path}")
    
    # カメラ接続を解除
    camera.Disconnect()
    print("Camera disconnected")

except Exception as e:
    print(f"Error occurred: {str(e)}")
    if 'camera' in locals() and camera is not None:
        try:
            camera.Disconnect()
            print("Camera disconnected after error")
        except:
            pass

# 3. detecting target source position


# 4. calculating distance between target source and slit center


# 5. converting distance unit to radec

# 6. transport fixed-radec-infomation(json)